# Lab Task 02 — Effect of Image Filtering on Skin-Lesion Classification

**Course:** Introduction to Computer Vision
**Dataset:** ISIC Skin Cancer (9 classes) — `nodoubttome/skin-cancer9-classesisic`
**Models (best 3 from Lab Activity 1 / Task 01):**
1. ResNet101 — 85.33% accuracy in Task 01
2. DenseNet121 — 84.67% accuracy in Task 01
3. ResNet50 — 83.33% accuracy in Task 01

**Filters compared against the unfiltered baseline:** Average/Mean, Gaussian, Median, Sharpening, Sobel edge

**Notebook sections** (matching the assignment's required structure):
1. Setup
2. Dataset preparation
3. Image filtering
4. Model loading
5. Training & evaluation utilities
6. Baseline experiment (no filter)
7. Filtering experiments
8. Visualization
9. Comparative analysis
10. Questions to answer

> Run cells top to bottom. Everything is driven by the `CONFIG` dict in Section 1 — change it there, not throughout the notebook.


## 1. Setup

In [ ]:
# Install/upgrade the few packages Colab doesn't already have pinned correctly.
!pip install -q kagglehub torchmetrics


In [ ]:
import os
import cv2
import glob
import random
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4) is strongly recommended.")


In [ ]:
# ------------------------------------------------------------------
# CONFIG — the single place to tune the experiment
# ------------------------------------------------------------------
CONFIG = {
    "IMG_SIZE": 224,
    "IMAGES_PER_CLASS": 300,     # balanced subset per class (train+val+test combined), as requested
    "TRAIN_FRAC": 0.70,
    "VAL_FRAC": 0.15,
    "TEST_FRAC": 0.15,
    "BATCH_SIZE": 32,
    "EPOCHS": 8,                 # keep modest for Colab GPU time; raise if you have time/budget
    "LR": 1e-4,
    "WEIGHT_DECAY": 1e-4,
    "SEED": 42,
    "NUM_WORKERS": 2,
}

MODEL_NAMES = ["ResNet101", "DenseNet121", "ResNet50"]   # best 3 models from Task 01
FILTER_NAMES = ["No Filter", "Average", "Gaussian", "Median", "Sharpening", "Sobel"]

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
torch.manual_seed(CONFIG["SEED"])
torch.cuda.manual_seed_all(CONFIG["SEED"])


## 2. Dataset preparation

Downloads the dataset, locates the `Train`/`Test` class folders, builds a single
dataframe of `(filepath, label)`, shows the class distribution, then samples a
**balanced subset** (`IMAGES_PER_CLASS` images per class) and splits it into
train/val/test. The same split (by filepath) is reused, unfiltered, for every
model x filter combination, so comparisons are fair.


In [ ]:
# Download the dataset (as provided)
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)


In [ ]:
# Auto-discover class folders under Train/ and Test/ (structure can vary slightly
# depending on how kagglehub unpacks the archive, so we search for it).
def find_split_dirs(root):
    train_dir, test_dir = None, None
    for dirpath, dirnames, filenames in os.walk(root):
        low = [d.lower() for d in dirnames]
        if "train" in low and train_dir is None:
            train_dir = os.path.join(dirpath, dirnames[low.index("train")])
        if "test" in low and test_dir is None:
            test_dir = os.path.join(dirpath, dirnames[low.index("test")])
    return train_dir, test_dir

train_dir, test_dir = find_split_dirs(path)
print("Train dir:", train_dir)
print("Test dir :", test_dir)

def list_images(split_dir):
    records = []
    if split_dir is None:
        return records
    for class_name in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        for fname in os.listdir(class_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                records.append({"filepath": os.path.join(class_path, fname), "label": class_name})
    return records

records = list_images(train_dir) + list_images(test_dir)
df_all = pd.DataFrame(records)
print("Total images found:", len(df_all))
print("Classes:", sorted(df_all['label'].unique()))
df_all['label'].value_counts()


In [ ]:
# Class distribution (full dataset, before balancing)
plt.figure(figsize=(10, 4))
order = df_all['label'].value_counts().index
sns.countplot(data=df_all, y='label', order=order)
plt.title("Full dataset class distribution (ISIC 9-class)")
plt.xlabel("Number of images")
plt.ylabel("Class")
plt.tight_layout()
plt.show()


In [ ]:
# Build a BALANCED subset: IMAGES_PER_CLASS images per class (or all available
# if a class has fewer), then a stratified train/val/test split reused everywhere.
def build_balanced_subset(df, per_class, seed):
    parts = []
    for label, group in df.groupby('label'):
        n = min(per_class, len(group))
        parts.append(group.sample(n=n, random_state=seed))
    return pd.concat(parts).reset_index(drop=True)

df_bal = build_balanced_subset(df_all, CONFIG["IMAGES_PER_CLASS"], CONFIG["SEED"])
print("Balanced subset size:", len(df_bal))
print(df_bal['label'].value_counts())

classes = sorted(df_bal['label'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
num_classes = len(classes)
df_bal['label_idx'] = df_bal['label'].map(class_to_idx)

plt.figure(figsize=(10, 4))
sns.countplot(data=df_bal, y='label', order=classes)
plt.title(f"Balanced subset class distribution ({CONFIG['IMAGES_PER_CLASS']} imgs/class target)")
plt.xlabel("Number of images")
plt.ylabel("Class")
plt.tight_layout()
plt.show()


In [ ]:
# Stratified split, reused identically across every model/filter combination
train_df, temp_df = train_test_split(
    df_bal, test_size=(1 - CONFIG["TRAIN_FRAC"]),
    stratify=df_bal['label_idx'], random_state=CONFIG["SEED"]
)
rel_val = CONFIG["VAL_FRAC"] / (CONFIG["VAL_FRAC"] + CONFIG["TEST_FRAC"])
val_df, test_df = train_test_split(
    temp_df, test_size=(1 - rel_val),
    stratify=temp_df['label_idx'], random_state=CONFIG["SEED"]
)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")


## 3. Image filtering

Each filter is implemented with OpenCV and applied to the **raw RGB image
before resizing/normalization**, so every model sees the same filtered pixels.
Sobel produces a single-channel edge map, which we replicate to 3 channels so
it can still be fed into an ImageNet-pretrained backbone.


In [ ]:
def apply_filter(img_rgb, filter_name):
    """img_rgb: HxWx3 uint8 numpy array (RGB). Returns HxWx3 uint8 numpy array."""
    if filter_name == "No Filter":
        return img_rgb

    if filter_name == "Average":
        return cv2.blur(img_rgb, (5, 5))

    if filter_name == "Gaussian":
        return cv2.GaussianBlur(img_rgb, (5, 5), sigmaX=0)

    if filter_name == "Median":
        return cv2.medianBlur(img_rgb, 5)

    if filter_name == "Sharpening":
        kernel = np.array([[0, -1, 0],
                            [-1, 5, -1],
                            [0, -1, 0]])
        return cv2.filter2D(img_rgb, -1, kernel)

    if filter_name == "Sobel":
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        mag = np.sqrt(sx ** 2 + sy ** 2)
        mag = np.uint8(255 * mag / (mag.max() + 1e-8))
        return cv2.cvtColor(mag, cv2.COLOR_GRAY2RGB)

    raise ValueError(f"Unknown filter: {filter_name}")


In [ ]:
# Visualize the original image + every filter on a handful of sample images
sample_rows = df_bal.sample(n=3, random_state=CONFIG["SEED"])
fig, axes = plt.subplots(len(sample_rows), len(FILTER_NAMES), figsize=(3 * len(FILTER_NAMES), 3 * len(sample_rows)))

for row_i, (_, row) in enumerate(sample_rows.iterrows()):
    img = np.array(Image.open(row['filepath']).convert("RGB"))
    for col_i, fname in enumerate(FILTER_NAMES):
        filtered = apply_filter(img, fname)
        ax = axes[row_i, col_i] if len(sample_rows) > 1 else axes[col_i]
        ax.imshow(filtered)
        ax.axis("off")
        if row_i == 0:
            ax.set_title(fname)
        if col_i == 0:
            ax.set_ylabel(row['label'], fontsize=9)

plt.suptitle("Original and filtered image examples")
plt.tight_layout()
plt.show()


In [ ]:
# Dataset class: loads an image, applies the chosen filter, then the standard
# ImageNet-pretrained preprocessing (resize + normalize).
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, filter_name, img_size, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.filter_name = filter_name
        self.train = train
        aug = [transforms.RandomHorizontalFlip(), transforms.RandomRotation(15)] if train else []
        self.to_tensor = transforms.Compose(
            aug + [
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['filepath']).convert("RGB"))
        img = apply_filter(img, self.filter_name)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img_pil = Image.fromarray(img)
        tensor = self.to_tensor(img_pil)
        return tensor, row['label_idx']


def make_loaders(filter_name):
    train_ds = SkinLesionDataset(train_df, filter_name, CONFIG["IMG_SIZE"], train=True)
    val_ds = SkinLesionDataset(val_df, filter_name, CONFIG["IMG_SIZE"], train=False)
    test_ds = SkinLesionDataset(test_df, filter_name, CONFIG["IMG_SIZE"], train=False)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True,
                               num_workers=CONFIG["NUM_WORKERS"])
    val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                             num_workers=CONFIG["NUM_WORKERS"])
    test_loader = DataLoader(test_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                              num_workers=CONFIG["NUM_WORKERS"])
    return train_loader, val_loader, test_loader


## 4. Model loading

Builds the three pretrained backbones identified as the best performers in
Task 01 (ResNet101, DenseNet121, ResNet50), each with its classification head
replaced for `num_classes`, and fully fine-tuned.


In [ ]:
def build_model(model_name, num_classes):
    if model_name == "ResNet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif model_name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m.to(device)


## 5. Training & evaluation utilities

Shared functions used by every experiment (baseline + each filter, for each
model), so training settings and evaluation procedure stay identical across
runs — as the assignment requires for a fair comparison.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_loss_acc(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def full_evaluation(model, loader, num_classes):
    """Runs inference once and returns every metric the assignment asks for."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    precision_pc, recall_pc, f1_pc, support_pc = precision_recall_fscore_support(
        y_true, y_pred, labels=range(num_classes), zero_division=0
    )
    precision_macro = precision_pc.mean()
    recall_macro = recall_pc.mean()
    f1_macro = f1_pc.mean()
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))

    try:
        y_true_bin = label_binarize(y_true, classes=range(num_classes))
        auc_macro = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
    except ValueError:
        auc_macro = float("nan")

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "per_class_precision": precision_pc,
        "per_class_recall": recall_pc,
        "per_class_f1": f1_pc,
        "confusion_matrix": cm,
        "auc_macro": auc_macro,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def run_experiment(model_name, filter_name):
    """Trains one model on one filter setting and returns history + test metrics."""
    print(f"\n===== {model_name} | {filter_name} =====")
    train_loader, val_loader, test_loader = make_loaders(filter_name)
    model = build_model(model_name, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = -1, None

    for epoch in range(CONFIG["EPOCHS"]):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']}  "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = full_evaluation(model, test_loader, num_classes)
    print(f"TEST  acc={test_metrics['accuracy']:.4f}  macro-F1={test_metrics['f1_macro']:.4f}  "
          f"bal-acc={test_metrics['balanced_accuracy']:.4f}  AUC={test_metrics['auc_macro']:.4f}")

    return {
        "model_name": model_name,
        "filter_name": filter_name,
        "history": history,
        "test_metrics": test_metrics,
    }


## 6. Baseline experiment (no filter)

Trains each of the three models on the **original, unfiltered** images. This
establishes the reference point every filtered run is compared against.


In [ ]:
all_results = {}   # key: (model_name, filter_name) -> result dict from run_experiment

for model_name in MODEL_NAMES:
    result = run_experiment(model_name, "No Filter")
    all_results[(model_name, "No Filter")] = result


## 7. Filtering experiments

Repeats the identical training/evaluation procedure for each of the 5 filters,
for each of the 3 models — same split, same hyperparameters, same code path as
the baseline, so only the filter changes.


In [ ]:
filters_to_run = [f for f in FILTER_NAMES if f != "No Filter"]

for model_name in MODEL_NAMES:
    for filter_name in filters_to_run:
        result = run_experiment(model_name, filter_name)
        all_results[(model_name, filter_name)] = result


## 8. Visualization

Training/validation accuracy & loss curves, confusion matrices, and ROC curves
for every model x filter combination.


In [ ]:
# Accuracy & loss curves — one figure per model, all filters overlaid
for model_name in MODEL_NAMES:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for filter_name in FILTER_NAMES:
        h = all_results[(model_name, filter_name)]["history"]
        axes[0].plot(h["val_acc"], label=filter_name)
        axes[1].plot(h["val_loss"], label=filter_name)
    axes[0].set_title(f"{model_name} — Validation accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend(fontsize=8)
    axes[1].set_title(f"{model_name} — Validation loss")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# Confusion matrices — grid of (model x filter)
fig, axes = plt.subplots(len(MODEL_NAMES), len(FILTER_NAMES),
                          figsize=(3.2 * len(FILTER_NAMES), 3.2 * len(MODEL_NAMES)))

for i, model_name in enumerate(MODEL_NAMES):
    for j, filter_name in enumerate(FILTER_NAMES):
        cm = all_results[(model_name, filter_name)]["test_metrics"]["confusion_matrix"]
        ax = axes[i, j]
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
        disp.plot(ax=ax, colorbar=False, xticks_rotation=90)
        ax.set_title(f"{model_name}\n{filter_name}", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ROC curves (one-vs-rest, macro) — one figure per model, all filters overlaid
for model_name in MODEL_NAMES:
    plt.figure(figsize=(6, 5))
    for filter_name in FILTER_NAMES:
        tm = all_results[(model_name, filter_name)]["test_metrics"]
        y_true_bin = label_binarize(tm["y_true"], classes=range(num_classes))
        fpr, tpr, _ = roc_curve(y_true_bin.ravel(), tm["y_prob"].ravel())
        plt.plot(fpr, tpr, label=f"{filter_name} (AUC={tm['auc_macro']:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.title(f"{model_name} — micro-averaged ROC across filters")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## 9. Comparative analysis

Assembles every run into the comparison table required by the assignment,
plus per-class precision/recall/F1, and bar charts comparing filters within a
model and models within a filter.


In [ ]:
rows = []
for model_name in MODEL_NAMES:
    for filter_name in FILTER_NAMES:
        tm = all_results[(model_name, filter_name)]["test_metrics"]
        rows.append({
            "Model": model_name,
            "Filter": filter_name,
            "Accuracy": round(tm["accuracy"], 4),
            "Precision (macro)": round(tm["precision_macro"], 4),
            "Recall (macro)": round(tm["recall_macro"], 4),
            "F1-score (macro)": round(tm["f1_macro"], 4),
            "Balanced Accuracy": round(tm["balanced_accuracy"], 4),
            "Macro-F1": round(tm["f1_macro"], 4),
            "AUC": round(tm["auc_macro"], 4),
        })

results_table = pd.DataFrame(rows)
results_table = results_table.sort_values(["Model", "Filter"]).reset_index(drop=True)
results_table


In [ ]:
# Save the comparison table (this is the "Required Experimental Results" table)
results_table.to_csv("comparison_table.csv", index=False)
print("Saved comparison_table.csv")
results_table


In [ ]:
# Per-class precision/recall/F1 for every model x filter combination
per_class_rows = []
for model_name in MODEL_NAMES:
    for filter_name in FILTER_NAMES:
        tm = all_results[(model_name, filter_name)]["test_metrics"]
        for cls_idx, cls_name in idx_to_class.items():
            per_class_rows.append({
                "Model": model_name,
                "Filter": filter_name,
                "Class": cls_name,
                "Precision": round(tm["per_class_precision"][cls_idx], 4),
                "Recall": round(tm["per_class_recall"][cls_idx], 4),
                "F1-score": round(tm["per_class_f1"][cls_idx], 4),
            })

per_class_table = pd.DataFrame(per_class_rows)
per_class_table.to_csv("per_class_metrics.csv", index=False)
print("Saved per_class_metrics.csv")
per_class_table.head(15)


In [ ]:
# Bar chart: Macro-F1 by filter, one group of bars per model
plt.figure(figsize=(10, 5))
pivot = results_table.pivot(index="Filter", columns="Model", values="F1-score (macro)")
pivot = pivot.reindex(FILTER_NAMES)
pivot.plot(kind="bar", ax=plt.gca())
plt.title("Macro-F1 by filter, per model")
plt.ylabel("Macro-F1")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart: Balanced accuracy by filter, one group of bars per model
plt.figure(figsize=(10, 5))
pivot2 = results_table.pivot(index="Filter", columns="Model", values="Balanced Accuracy")
pivot2 = pivot2.reindex(FILTER_NAMES)
pivot2.plot(kind="bar", ax=plt.gca())
plt.title("Balanced accuracy by filter, per model")
plt.ylabel("Balanced Accuracy")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Change relative to each model's own "No Filter" baseline — this is what
# question "Which filter produces the greatest change...?" is really asking.
baseline_f1 = results_table[results_table["Filter"] == "No Filter"].set_index("Model")["F1-score (macro)"]
results_table["Delta Macro-F1 vs baseline"] = results_table.apply(
    lambda r: round(r["F1-score (macro)"] - baseline_f1[r["Model"]], 4), axis=1
)
results_table.sort_values(["Model", "Delta Macro-F1 vs baseline"])


## 10. Questions to answer

Use the tables and plots above to fill these in with your actual numbers —
don't just answer generically.

1. **Which three pretrained models performed best in Lab Activity 1?**
   ResNet101, DenseNet121, and ResNet50 (see Task 01 results) — used here.

2. **How does filtering affect each of the three models?**
   *(Read off `results_table`: for each model, compare each filter row to its
   "No Filter" row using the `Delta Macro-F1 vs baseline` column.)*

3. **Which filter produces the greatest change compared with the unfiltered baseline?**
   *(`results_table.reindex(results_table['Delta Macro-F1 vs baseline'].abs().sort_values(ascending=False).index)` — largest absolute delta.)*

4. **Does the effect of a filter remain consistent across all three models?**
   *(Compare the sign/magnitude of `Delta Macro-F1 vs baseline` for the same filter across the 3 models.)*

5. **Does filtering improve or decrease macro-F1 and balanced accuracy?**
   *(Read directly from `results_table`.)*

6. **Which lesion classes are most affected by filtering?**
   *(Compare `per_class_table` F1-scores for the same class across filters, within a model.)*

7. **Why might smoothing remove useful lesion texture or morphological information?**
   Averaging/Gaussian/Median filters blur high-frequency detail — border
   irregularity, pigment network texture, fine structures — that dermoscopic
   classification relies on, effectively acting as a low-pass filter on the
   very cues that distinguish lesion types.

8. **Why might sharpening or edge detection help or hurt classification?**
   Sharpening can amplify lesion borders and texture, which may help when
   those boundaries are diagnostic, but it also amplifies noise and artifacts
   (hair, ruler marks) which can hurt. Sobel keeps only edge magnitude and
   discards color/texture information the network normally relies on, which
   usually hurts overall accuracy even if it emphasizes lesion shape.

9. **What is the difference between convolution and correlation?**
   Convolution flips the kernel both horizontally and vertically before
   sliding it over the image, whereas (cross-)correlation slides the
   (unflipped) kernel directly. For a symmetric kernel (e.g. a Gaussian) the
   two operations are identical; for asymmetric kernels (e.g. a Sobel kernel)
   they differ. Most deep-learning frameworks, including the "convolution"
   layers in the CNNs used here, actually implement cross-correlation for
   simplicity — the kernel weights are learned, so flipping makes no
   practical difference to what the network can learn.

10. **Based on your results, explain the relationship between classical image
    processing and deep-learning-based feature extraction.**
    *(Write this after reviewing your own numbers — but generally: classical
    filters are fixed, hand-designed operators applied before the network
    sees the image, while CNN convolutional layers learn their own filters
    end-to-end from data. Pre-filtering can help when it removes irrelevant
    noise the network would otherwise have to learn to ignore, but it can
    also destroy information the network would have learned to exploit,
    which is why the effect is filter- and architecture-dependent rather than
    uniformly positive or negative.)*


## Code submission checklist (per the assignment)

- [x] Complete experimental code developed in this Colab/Jupyter notebook
- [x] Organized into clearly labeled sections: Dataset preparation, Model
      loading, Baseline experiment, Image filtering, Training, Evaluation,
      Visualization, Comparative analysis
- [ ] Push this notebook + a README to the GitHub repo under `Lab 02/`
      (see the suggested `README.md` below)


### Suggested `README.md` for the GitHub `Lab 02/` folder

```markdown
# Lab 02 — Effect of Image Filtering on Skin-Lesion Classification

## Contents
- `Lab02_Skin_Lesion_Filtering.ipynb` — full experiment notebook
- `comparison_table.csv` — Model x Filter results (accuracy, precision, recall, F1, macro-F1, balanced accuracy, AUC)
- `per_class_metrics.csv` — per-class precision/recall/F1 for every model x filter run

## How to run
1. Open the notebook in Google Colab.
2. Runtime > Change runtime type > GPU (T4 or better).
3. Run all cells top to bottom. The `CONFIG` dict in Section 1 controls
   image size, balanced-subset size, batch size, epochs, and learning rate.
4. The dataset is downloaded automatically via `kagglehub`
   (`nodoubttome/skin-cancer9-classesisic`) in Section 2 — no manual upload needed.
5. Results are written to `comparison_table.csv` and `per_class_metrics.csv`
   in the Colab working directory; download them from the Files pane if you
   want to keep them alongside the notebook.

## Models
ResNet101, DenseNet121, ResNet50 — the top 3 performers from Lab 01 (Task 01).

## Filters
No Filter (baseline), Average/Mean, Gaussian, Median, Sharpening, Sobel edge — each applied to the raw RGB image before resizing/normalization.
```
